# 뉴스 데이터 전처리 및 TF-IDF 키워드 추출

## 처리 내용
1. `keyword`, `link` 컬럼 삭제
2. `date` 컬럼을 정규식으로 `YYYY-MM-DD` 형식으로 변환
3. `title`과 `content`에서 TF-IDF로 키워드 추출하여 새로운 `keyword` 컬럼 생성

In [73]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from pathlib import Path

In [74]:
# 경로 설정
DATA_DIR = Path(r'd:\0.Sogang\동아리 및 학회\Insight\2025-2\2차 인사이콘\25-2-Insightcon\data\raw\news')
INPUT_FILE = DATA_DIR / 'Semiconductor_Final_Integrated_20251220_1141.csv'
OUTPUT_FILE = DATA_DIR / 'News_processed.csv'

# TF-IDF 파라미터
MAX_KEYWORDS = 10  # 추출할 최대 키워드 개수
MIN_DF = 2  # 최소 문서 빈도
MAX_DF = 0.9  # 최대 문서 빈도 (너무 흔한 단어 제외)
NGRAM_RANGE = (1, 2)  # unigram과 bigram 모두 사용

In [75]:
# 데이터 로드
print(f"데이터 로드 중: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE)
print(f"원본 데이터 shape: {df.shape}")
print(f"원본 컬럼: {df.columns.tolist()}")
print(f"\n첫 3행 샘플:")
df.head(3)

데이터 로드 중: d:\0.Sogang\동아리 및 학회\Insight\2025-2\2차 인사이콘\25-2-Insightcon\data\raw\news\Semiconductor_Final_Integrated_20251220_1141.csv
원본 데이터 shape: (4815, 5)
원본 컬럼: ['keyword', 'title', 'date', 'link', 'content']

첫 3행 샘플:


,keyword,title,date,link,content
0,DRAM 고정가,[주가동향] “삼성전자 선호 전략 유지… 반도체 업황 반등 조짐” [하...,2025-08-19 09:48,https://n.news.naver.com/mnews/article/123/000...,"8월 10일 기준 메모리 수출 22%↑, NAND도 상승 전환 파운드리 가치 재평가..."
1,DRAM 고정가,"DDR4 포기 삼성·SK하이닉스, 막대한 수익 기회 상실",2025-06-24 12:34,https://n.news.naver.com/mnews/article/123/000...,최근 급등한 DDR4 가격은 삼성전자와 SK하이닉스가 전략적으로 선택한 '고성능 집...
2,DRAM 고정가,[주간 추천주] 실적 개선 본격화…눈 여겨볼 종목은?,2024-03-17 12:01,https://n.news.naver.com/mnews/article/031/000...,한화오션·삼성생명·크래프톤 '주목' 증권가에서 실적 개선이 본격화될 회사에 주목할 ...


In [76]:
# 1. keyword, link 컬럼 삭제
columns_to_drop = ['keyword', 'link']
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]

if existing_columns_to_drop:
    df = df.drop(columns=existing_columns_to_drop)
    print(f"삭제된 컬럼: {existing_columns_to_drop}")
else:
    print("삭제할 컬럼이 없습니다.")

print(f"컬럼 삭제 후 shape: {df.shape}")
print(f"남은 컬럼: {df.columns.tolist()}")

삭제된 컬럼: ['keyword', 'link']
컬럼 삭제 후 shape: (4815, 3)
남은 컬럼: ['title', 'date', 'content']


In [77]:
# 2. date 컬럼 정규화 (YYYY-MM-DD만 추출)
def normalize_date(date_str):
    """날짜 문자열에서 YYYY-MM-DD 형식만 추출"""
    if pd.isna(date_str):
        return None
    
    # 정규식: YYYY-MM-DD 패턴 추출
    match = re.search(r'(\d{4}-\d{2}-\d{2})', str(date_str))
    return match.group(1) if match else None

if 'date' in df.columns:
    print("\n날짜 정규화 전 샘플:")
    print(df['date'].head(5).tolist())
    
    df['date'] = df['date'].apply(normalize_date)
    
    print("\n날짜 정규화 후 샘플:")
    print(df['date'].head(5).tolist())
    print(f"\nNull 날짜 개수: {df['date'].isna().sum()}")
else:
    print("date 컬럼이 존재하지 않습니다.")


날짜 정규화 전 샘플:
['2025-08-19 09:48', '2025-06-24 12:34', '2024-03-17 12:01', '2024-03-16 09:56', '2023-09-10 07:12']

날짜 정규화 후 샘플:
['2025-08-19', '2025-06-24', '2024-03-17', '2024-03-16', '2023-09-10']

Null 날짜 개수: 0


In [78]:
df['date']

0       2025-08-19
1       2025-06-24
2       2024-03-17
3       2024-03-16
4       2023-09-10
           ...    
4810    2024-06-17
4811    2024-06-14
4812    2024-06-13
4813    2024-06-12
4814    2024-06-12
Name: date, Length: 4815, dtype: object

In [79]:
# 3. TF-IDF 키워드 추출 함수
def extract_keywords_tfidf(texts, max_keywords=MAX_KEYWORDS):
    """
    TF-IDF를 사용하여 텍스트에서 키워드 추출
    
    Args:
        texts: 텍스트 리스트
        max_keywords: 문서당 추출할 최대 키워드 개수
    
    Returns:
        각 문서의 키워드 리스트
    """
    # 결측값 처리
    processed_texts = [str(text) if pd.notna(text) else "" for text in texts]
    
    # TF-IDF 벡터라이저 초기화
    vectorizer = TfidfVectorizer(
        max_features=1000,
        min_df=MIN_DF,
        max_df=MAX_DF,
        ngram_range=NGRAM_RANGE,
        stop_words='english',  # 영어 불용어 제거
        lowercase=True
    )
    
    try:
        # TF-IDF 행렬 생성
        tfidf_matrix = vectorizer.fit_transform(processed_texts)
        feature_names = vectorizer.get_feature_names_out()
        
        # 각 문서의 상위 키워드 추출
        keywords_list = []
        for doc_idx in range(tfidf_matrix.shape[0]):
            # 해당 문서의 TF-IDF 점수
            tfidf_scores = tfidf_matrix[doc_idx].toarray().flatten()
            
            # 상위 키워드 인덱스
            top_indices = tfidf_scores.argsort()[-max_keywords:][::-1]
            
            # 키워드 추출 (점수가 0보다 큰 것만)
            keywords = [
                feature_names[idx] 
                for idx in top_indices 
                if tfidf_scores[idx] > 0
            ]
            
            keywords_list.append(', '.join(keywords) if keywords else '')
        
        return keywords_list
    
    except Exception as e:
        print(f"TF-IDF 처리 중 오류: {e}")
        return [''] * len(processed_texts)

In [80]:
# title과 content 결합
print("\ntitle과 content 결합 중...")
df['combined_text'] = df.apply(
    lambda row: f"{row.get('title', '')} {row.get('content', '')}".strip(),
    axis=1
)

print(f"결합된 텍스트 샘플:")
print(df['combined_text'].iloc[0][:200] + "...")


title과 content 결합 중...
결합된 텍스트 샘플:
[주가동향] “삼성전자 선호 전략 유지… 반도체 업황 반등 조짐” [하... 8월 10일 기준 메모리 수출 22%↑, NAND도 상승 전환 파운드리 가치 재평가·테슬라 고객사 확보 ‘긍정적’ DRAM·NAND 동반 성장, 8월 잠정치서 개선세 검증 메모리 반도체 가격이 반등 조짐을 보이는 가운데, 수출 회복세까지 더해져 국내 대형 반도체주가 업종 내 아웃퍼...


In [81]:
# TF-IDF 키워드 추출
print("\nTF-IDF 키워드 추출 중...")
df['keyword'] = extract_keywords_tfidf(df['combined_text'].tolist())

print(f"\n키워드 추출 완료!")
print(f"키워드가 있는 행: {(df['keyword'] != '').sum()}")
print(f"키워드가 없는 행: {(df['keyword'] == '').sum()}")

# 샘플 확인
print("\n키워드 추출 샘플 (첫 5행):")
for idx in range(min(5, len(df))):
    print(f"\n[{idx}] Title: {df.iloc[idx].get('title', '')[:50]}...")
    print(f"    Keywords: {df.iloc[idx]['keyword']}")


TF-IDF 키워드 추출 중...

키워드 추출 완료!
키워드가 있는 행: 4815
키워드가 없는 행: 0

키워드 추출 샘플 (첫 5행):

[0] Title: [주가동향] “삼성전자 선호 전략 유지… 반도체 업황 반등 조짐” [하......
    Keywords: dram, nand, 8월, 수출, 가격, ddr5, 메모리, 22, 보였다, 삼성전자

[1] Title: DDR4 포기 삼성·SK하이닉스, 막대한 수익 기회 상실...
    Keywords: ddr4, 2025년, dram, 100, 점에서, 삼성전자와, sk하이닉스는, 반면, ddr5, 중심의

[2] Title: [주간 추천주] 실적 개선 본격화…눈 여겨볼 종목은?...
    Keywords: 기대된다, 실적, 따른, 모바일, 분석했다, 대해, 판매, 전망된다, 전망했다, 올해

[3] Title: [하나證 주간추천주]SK하이닉스·삼성생명·에쓰오일...
    Keywords: 전망, sk하이닉스, 기대, 밸류업, 실적, dram, nand, 예정인, 이데일리, 판매

[4] Title: 국내경기 ‘상저하중’?…상장사 3분기 이익 증가 전망에도 연간으론 -......
    Keywords: 3분기, 경기, 영업이익, 영업이익은, 영업이익이, 코스피, 대비, 크게, 실적, 동기 대비


In [82]:
# 임시 컬럼 제거
df = df.drop(columns=['combined_text'])

print("\n최종 데이터 shape:", df.shape)
print("최종 컬럼:", df.columns.tolist())
df.head()


최종 데이터 shape: (4815, 4)
최종 컬럼: ['title', 'date', 'content', 'keyword']


,title,date,content,keyword
0,[주가동향] “삼성전자 선호 전략 유지… 반도체 업황 반등 조짐” [하...,2025-08-19,"8월 10일 기준 메모리 수출 22%↑, NAND도 상승 전환 파운드리 가치 재평가...","dram, nand, 8월, 수출, 가격, ddr5, 메모리, 22, 보였다, 삼성전자"
1,"DDR4 포기 삼성·SK하이닉스, 막대한 수익 기회 상실",2025-06-24,최근 급등한 DDR4 가격은 삼성전자와 SK하이닉스가 전략적으로 선택한 '고성능 집...,"ddr4, 2025년, dram, 100, 점에서, 삼성전자와, sk하이닉스는, 반..."
2,[주간 추천주] 실적 개선 본격화…눈 여겨볼 종목은?,2024-03-17,한화오션·삼성생명·크래프톤 '주목' 증권가에서 실적 개선이 본격화될 회사에 주목할 ...,"기대된다, 실적, 따른, 모바일, 분석했다, 대해, 판매, 전망된다, 전망했다, 올해"
3,[하나證 주간추천주]SK하이닉스·삼성생명·에쓰오일,2024-03-16,[이데일리 박순엽 기자] △SK하이닉스(000660) -오는 2분기부터 출하 예정인...,"전망, sk하이닉스, 기대, 밸류업, 실적, dram, nand, 예정인, 이데일리..."
4,국내경기 ‘상저하중’?…상장사 3분기 이익 증가 전망에도 연간으론 -...,2023-09-10,3분기 한전 흑자전환·반도체 업황 반등 영업이익 1위 현대차…2위 삼성전자 탈환 하...,"3분기, 경기, 영업이익, 영업이익은, 영업이익이, 코스피, 대비, 크게, 실적, ..."


In [83]:
# 결과 저장
print(f"\n결과 저장 중: {OUTPUT_FILE}")
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"저장 완료!")
print(f"\n처리된 데이터:")
print(f"  - 총 행 수: {len(df)}")
print(f"  - 컬럼: {df.columns.tolist()}")
print(f"  - 파일 크기: {OUTPUT_FILE.stat().st_size / 1024 / 1024:.2f} MB")


결과 저장 중: d:\0.Sogang\동아리 및 학회\Insight\2025-2\2차 인사이콘\25-2-Insightcon\data\raw\news\News_processed.csv
저장 완료!

처리된 데이터:
  - 총 행 수: 4815
  - 컬럼: ['title', 'date', 'content', 'keyword']
  - 파일 크기: 19.24 MB


In [84]:
df

,title,date,content,keyword
0,[주가동향] “삼성전자 선호 전략 유지… 반도체 업황 반등 조짐” [하...,2025-08-19,"8월 10일 기준 메모리 수출 22%↑, NAND도 상승 전환 파운드리 가치 재평가...","dram, nand, 8월, 수출, 가격, ddr5, 메모리, 22, 보였다, 삼성전자"
1,"DDR4 포기 삼성·SK하이닉스, 막대한 수익 기회 상실",2025-06-24,최근 급등한 DDR4 가격은 삼성전자와 SK하이닉스가 전략적으로 선택한 '고성능 집...,"ddr4, 2025년, dram, 100, 점에서, 삼성전자와, sk하이닉스는, 반..."
2,[주간 추천주] 실적 개선 본격화…눈 여겨볼 종목은?,2024-03-17,한화오션·삼성생명·크래프톤 '주목' 증권가에서 실적 개선이 본격화될 회사에 주목할 ...,"기대된다, 실적, 따른, 모바일, 분석했다, 대해, 판매, 전망된다, 전망했다, 올해"
3,[하나證 주간추천주]SK하이닉스·삼성생명·에쓰오일,2024-03-16,[이데일리 박순엽 기자] △SK하이닉스(000660) -오는 2분기부터 출하 예정인...,"전망, sk하이닉스, 기대, 밸류업, 실적, dram, nand, 예정인, 이데일리..."
4,국내경기 ‘상저하중’?…상장사 3분기 이익 증가 전망에도 연간으론 -...,2023-09-10,3분기 한전 흑자전환·반도체 업황 반등 영업이익 1위 현대차…2위 삼성전자 탈환 하...,"3분기, 경기, 영업이익, 영업이익은, 영업이익이, 코스피, 대비, 크게, 실적, ..."
...,...,...,...,...
4810,[정상균의 에브리싱] 반도체 ‘잃어버린 5년’,2024-06-17,'용인 클러스터’ 원대하나 갈등 많아 착공조차 못해 日 TSMC 공장 3년 걸려 며...,"반도체, 공장, tsmc, 되는, 보였다, 3년, 2027년, 2026년, 이상, 경기"
4811,HBM 공급 대란…SK하이닉스 日 생산 가능성은?,2024-06-14,"日, HBM테스트 등 소부장 기업 협업 장점 TSMC 日 공장 건설 단축 선례도 고...","hbm, 일본, 소부장, 현지, 증설, 건설, sk하이닉스는, 검토, 규모는, 연구개발"
4812,"[특집] 용인클러스터, 日TSMC 55배 규모… 美실리콘밸리 넘어선다",2024-06-13,"""세계 최대 반도체 생태계 도시로… 정부·정치권 전폭 지지 필요"" 삼성전자, 이동·...","반도체, 첨단, 규모, 구마모토, 시장은, 있다, 기업들이, 된다, 세계, 예정이다"
4813,[SBS Biz 포럼] 삼성 출신 고동진 日 반도체 재기에 소름…인수전 싸움...,2024-06-12,[고동진 국민의힘 의원이 12일 서울 중구 웨스틴 조선호텔에서 열린 SBS Biz ...,"반도체, 있습니다, 일본, 반도체는, 반도체 산업, 인프라, 지원, 것을, 보조금, 라며"
